# 01 — Data Audit

In [1]:
# Cell 1 — setup


import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / "src"))
from scorelab import config, data
con = data.connect()            # read-only DuckDB connection to data/lc.duckdb

In [2]:
# Cell 2 — sanity check
print(con.sql("SHOW TABLES"))
print(con.sql("SELECT COUNT(*) AS n, MIN(issue_d), MAX(issue_d) FROM accepted"))

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ accepted        │
│ rejected        │
│ v_accepted_asof │
│ v_split         │
│ v_target        │
└─────────────────┘

┌─────────┬──────────────┬──────────────┐
│    n    │ min(issue_d) │ max(issue_d) │
│  int64  │     date     │     date     │
├─────────┼──────────────┼──────────────┤
│ 2260668 │ 2007-06-01   │ 2018-12-01   │
└─────────┴──────────────┴──────────────┘



In [3]:
# Cell 3 — common imports and settings for this notebook
import json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
from scorelab import target, attributes as A, binning as B, scorecard as S, validation as V, fairness as F
from scorelab import challengers as C, reject_inference as R, monitoring as M, report as REP
MODELS = config.MODELS_DIR; MODELS.mkdir(exist_ok=True)
SAMPLE_FRAC = None     # None = full data (recommended for the fitting notebooks); e.g. 0.10 for a quick pass
cols = [r[0] for r in con.execute("DESCRIBE v_target").fetchall()]
assert "label" in cols, "v_target is the old stub: close this kernel and run `python scripts/refresh_views.py`"
def load_frame(splits=("train", "validate", "oot"), **kw):
    """Observable, non-excluded loans with attributes built; indeterminates dropped unless asked."""
    f = target.modeling_frame(con, splits=splits, sample_frac=SAMPLE_FRAC, **kw)
    return A.build_attributes(f)
MEANINGS = {a.name: a.meaning for a in A.REGISTRY}
DIRECTIONS = {a.name: a.direction for a in A.REGISTRY}

## What this notebook settles

Two decisions and a set of findings for the model document. **Decision A — scope:** which issue years, terms and application types the project models (the split in `config.py` uses 2013–2016 for fitting, 2017Q1–2018Q4 for monitoring). **Decision B — fields with structural gaps:** bureau fields that only exist for late vintages or are mostly missing in the training years are not attributes for a card that must be stable across years; list them here and NB03 drops them.

Write your reasoning in the decision cells before running the evidence below them.

In [4]:
# Evidence — what is in the database
display(data.row_counts(con))
display(con.sql("""
    SELECT year(issue_d) AS vintage_year, TRY_CAST(regexp_extract(term, '[0-9]+') AS INTEGER) AS term_months,
           COUNT(*) AS n, ROUND(AVG(loan_amnt), 0) AS mean_amount
    FROM accepted WHERE year(issue_d) >= 2012 GROUP BY 1, 2 ORDER BY 1, 2
""").df())
display(con.sql("SELECT loan_status, COUNT(*) AS n FROM accepted GROUP BY 1 ORDER BY 2 DESC").df())

,object,kind,rows
0,accepted,table,2260668
1,rejected,table,27648741
2,v_accepted_asof,view,2260668
3,v_split,view,2260668
4,v_target,view,2260668


,vintage_year,term_months,n,mean_amount
0,2012,36,43470,11682.0
1,2012,60,9897,21280.0
2,2013,36,100422,12667.0
3,2013,60,34392,20664.0
4,2014,36,162570,12586.0
5,2014,60,73059,19954.0
6,2015,36,283173,12807.0
7,2015,60,137922,20237.0
8,2016,36,323495,12768.0
9,2016,60,110912,20469.0


,loan_status,n
0,Fully Paid,1076751
1,Current,878317
2,Charged Off,268559
3,Late (31-120 days),21467
4,In Grace Period,8436
5,Late (16-30 days),4349
6,Does not meet the credit policy. Status:Fully ...,1988
7,Does not meet the credit policy. Status:Charge...,761
8,Default,40


In [5]:
# Evidence — field availability by vintage for every raw input the attribute registry reads
raw_inputs = sorted({c for a in A.REGISTRY for c in a.sources if c not in ("issue_d",)})
avail = data.missingness_by_vintage(raw_inputs, con).drop(index="n")   # share MISSING per field and year
display(avail.style.background_gradient(cmap="Reds", axis=None) if hasattr(avail, "style") else avail)
train_years = list(config.TRAIN_VINTAGES)
structural = avail[(avail.loc[:, train_years] > 0.95).any(axis=1)]            # absent for a whole training year
reporting_start = avail[(avail[train_years[0]] > 0.30) & (avail[train_years[-1]] < 0.05)]   # began mid-way through the first training year
print("Absent for an entire training year (structural):"); display(structural)
print("Reporting started during the first training year:"); display(reporting_start)
avail.to_csv(config.DOCS_DIR / "audit" / "field_missingness_by_vintage.csv")

vintage_year,2012,2013,2014,2015,2016,2017,2018
acc_now_delinq,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
acc_open_past_24mths,0.140400,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
all_util,1.000000,1.000000,1.000000,0.949200,0.000200,0.000200,0.000300
avg_cur_bal,0.519800,0.000000,0.000000,0.000000,0.000000,0.000000,0.000100
bc_open_to_buy,0.150300,0.007500,0.010400,0.009400,0.010700,0.012900,0.013300
bc_util,0.150800,0.007900,0.011100,0.010000,0.011100,0.013400,0.013700
chargeoff_within_12_mths,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
collections_12_mths_ex_med,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
delinq_2yrs,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
earliest_cr_line,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Absent for an entire training year (structural):


vintage_year,2012,2013,2014,2015,2016,2017,2018
all_util,1.0,1.0,1.0,0.9492,0.0002,0.0002,0.0003
il_util,1.0,1.0,1.0,0.9558,0.1322,0.1393,0.1632
inq_last_12m,1.0,1.0,1.0,0.9492,0.0001,0.0000,0.0000
max_bal_bc,1.0,1.0,1.0,0.9492,0.0001,0.0000,0.0000
mths_since_rcnt_il,1.0,1.0,1.0,0.9506,0.0271,0.0296,0.0372
open_il_24m,1.0,1.0,1.0,0.9492,0.0001,0.0000,0.0000
open_rv_24m,1.0,1.0,1.0,0.9492,0.0001,0.0000,0.0000


Reporting started during the first training year:


vintage_year,2012,2013,2014,2015,2016,2017,2018


In [6]:
# Evidence — when did the reporting-start fields begin? (by issue month of the first training year)
if len(reporting_start):
    probe = reporting_start.index[:6].tolist()
    exprs = ", ".join(f"ROUND(AVG(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END), 3) AS {c}" for c in probe)
    display(con.sql(f"SELECT issue_d, COUNT(*) AS n, {exprs} FROM v_accepted_asof WHERE year(issue_d) = {train_years[0]} GROUP BY 1 ORDER BY 1").df())

In [7]:
# Evidence — ranges and tails of the main numeric fields (caps and reporting artefacts)
key = ["revol_util", "bc_util", "all_util", "percent_bc_gt_75", "total_acc", "open_acc", "inq_last_6mths", "delinq_2yrs",
       "mths_since_last_delinq", "revol_bal", "tot_cur_bal", "avg_cur_bal", "bc_open_to_buy", "mo_sin_old_rev_tl_op"]
display(data.numeric_profile(key, con, years=tuple(config.TRAIN_VINTAGES) + (config.VALIDATE_VINTAGE, config.OOT_VINTAGE)))

,column,n_nonmissing,missing_share,min,p1,median,p99,max
0,revol_util,1225316,0.0005,0.0,2.4,54.0,98.50,892.3
1,bc_util,1213206,0.0104,0.0,0.1,65.6,101.00,339.6
2,all_util,455695,0.6283,0.0,10.0,61.0,103.00,204.0
3,percent_bc_gt_75,1213475,0.0102,0.0,0.0,50.0,100.00,100.0
4,total_acc,1225945,0.0000,2.0,6.0,23.0,61.00,176.0
5,open_acc,1225945,0.0000,0.0,3.0,11.0,29.00,97.0
6,inq_last_6mths,1225944,0.0000,0.0,0.0,0.0,4.00,6.0
7,delinq_2yrs,1225945,0.0000,0.0,0.0,0.0,4.00,39.0
8,mths_since_last_delinq,625102,0.4901,0.0,1.0,30.0,80.00,195.0
9,revol_bal,1225945,0.0000,0.0,359.0,11733.0,99577.00,2904836.0


In [8]:
# Evidence — categorical context fields and the incumbent score
for c in ("purpose", "application_type", "addr_state"):
    print(c); display(data.categorical_profile(c, con, top=12))
display(data.incumbent_coverage(con))

purpose


,level,n,share
0,debt_consolidation,1277877,0.5653
1,credit_card,516971,0.2287
2,home_improvement,150457,0.0666
3,other,139440,0.0617
4,major_purchase,50445,0.0223
5,medical,27488,0.0122
6,small_business,24689,0.0109
7,car,24013,0.0106
8,vacation,15525,0.0069
9,moving,15403,0.0068


application_type


,level,n,share
0,Individual,2139958,0.9466
1,Joint App,120710,0.0534


addr_state


,level,n,share
0,CA,314533,0.1391
1,NY,186389,0.0824
2,TX,186335,0.0824
3,FL,161991,0.0717
4,IL,91173,0.0403
5,NJ,83132,0.0368
6,PA,76939,0.0340
7,OH,75132,0.0332
8,GA,74196,0.0328
9,VA,62954,0.0278


,vintage_year,n,fico_missing,fico_min,fico_mean,fico_max,range_width
0,2012,53367,0.0,662.0,703.4,847.5,4.0
1,2013,134814,0.0,662.0,697.0,847.5,4.0
2,2014,235629,0.0,662.0,694.5,847.5,4.0
3,2015,421095,0.0,662.0,695.6,847.5,4.0
4,2016,434407,0.0,662.0,697.4,847.5,4.0
5,2017,443579,0.0,662.0,702.3,847.5,4.0
6,2018,495242,0.0,662.0,708.4,847.5,4.0


In [9]:
# Evidence — the rejected-applications file (through-the-door context for NB06)
display(con.sql("""
    SELECT year(application_date) AS year, COUNT(*) AS n_rejected,
           ROUND(AVG(CASE WHEN risk_score IS NULL OR risk_score = 0 THEN 1 ELSE 0 END), 4) AS risk_score_missing,
           ROUND(AVG(CASE WHEN risk_score > 0 THEN risk_score END), 1) AS mean_risk_score
    FROM rejected GROUP BY 1 ORDER BY 1
""").df())
display(R.through_the_door_profile(con, years=(2014, 2015, 2016)))

,year,n_rejected,risk_score_missing,mean_risk_score
0,2007,5274,0.0620,575.1
1,2008,25596,0.1381,598.9
2,2009,56991,0.1676,618.8
3,2010,112561,0.1555,641.1
4,2011,217792,0.1030,636.9
5,2012,337277,0.0685,641.6
6,2013,760942,0.0763,640.4
7,2014,1933700,0.1316,632.8
8,2015,2859379,0.8215,637.2
9,2016,4769874,0.7865,633.4


,fico_band,n_rejected,n_accepted,share_rejected,share_accepted,acceptance_rate
0,300,194.0,0.0,0.0001,0.0000,0.0000
1,320,158.0,0.0,0.0000,0.0000,0.0000
2,340,415.0,0.0,0.0001,0.0000,0.0000
3,360,224.0,0.0,0.0001,0.0000,0.0000
4,380,216.0,0.0,0.0001,0.0000,0.0000
5,400,440.0,0.0,0.0001,0.0000,0.0000
6,420,934.0,0.0,0.0003,0.0000,0.0000
7,440,1888.0,0.0,0.0006,0.0000,0.0000
8,460,3887.0,0.0,0.0012,0.0000,0.0000
9,480,10627.0,0.0,0.0033,0.0000,0.0000


### Decision A — Scope

Years, terms, application types. A block of bureau fields is unreported for the first part of 2012 and complete from the month it started: either begin the training window at that month (or at 2013, editing `config.TRAIN_VINTAGES` and re-running `scripts/refresh_views.py`), or drop those fields. Is a 60-month loan the same product as a 36-month loan for a 24-month score, or a segment? Are joint applications (none in the training years; 9,300 in 2015–2016) in or out?

**Your reasoning:**

**DECIDED 2026-10-31** (drafted by Claude from this notebook's full-data outputs, applied; pending final review by Siavash)

Train on **2013–2014**: `config.TRAIN_VINTAGES = (2013, 2014)`, then `make views`.

- **Why 2012 is out.** In this notebook's first full-data run, with 2012 still in training, a block of 21 bureau fields was about 52% missing for 2012 (`avg_cur_bal`, `tot_cur_bal`, `num_tl_op_past_12m` and others 0.5198) against 0.0000 from 2013. By issue month it was 100% missing from January to July 2012, 83.2% in August and 1.3% in September. Keeping 2012 would have made the missing bin mean "issued before September 2012".
- **Check after the change.** With 2013 as the first training year, the "reporting started during the first training year" table above is empty.
- **Cost.** 53,367 loans from 2012 leave training. Starting in September 2012 was considered and not taken, because it would need a month-level start date.

**Terms:** keep 36- and 60-month loans in one card. 60-month loans are 25.5% of 2013 (34,392 of 134,814), 31.0% of 2014 (73,059 of 235,629) and 25.5% of 2016 (110,912 of 434,407). Term is reported as a segment and is never an input: the score is bureau-only, and term is a product attribute that the lender combines with the score in its strategy.

**Application type:** joint applications (120,710 loans, 5.34% of all years) are excluded in NB02 (Decision 3).

### Decision B — Fields with structural gaps

From the missingness table: fields absent for an entire training year (the 2016-only block: `inq_last_12m`, `open_rv_24m`, `il_util`, `all_util`, `max_bal_bc`, `mths_since_rcnt_il`, `open_il_24m`) cannot be attributes of a card trained on 2013–2014. Fields that are missing because the consumer has no such event (`mths_since_last_delinq` = never delinquent) are *not* gaps: the missing bin is information. Decide the reporting-start block under Decision A. List the structural fields for NB03.

**Your reasoning and the list:**

**DECIDED 2026-10-31** (drafted by Claude from this notebook's full-data outputs, applied; pending final review by Siavash)

Structural fields for NB03: the 2016-only block, 7 fields: `all_util`, `il_util`, `inq_last_12m`, `max_bal_bc`, `mths_since_rcnt_il`, `open_il_24m`, `open_rv_24m`.

- They are 100% missing in 2013 and 2014 and 94.9–95.6% missing in 2015. From 2016 they are 0.0–2.7% missing (`il_util` 13.2%).
- With 2013 as the first training year, the >95% rule above flags exactly these 7.

**`mths_since_last_record` stays a candidate.** It was flagged in the first run only because it was 97.2% missing in 2012. It is 88.1% missing in 2013, 82.4% in 2014 and 80.9% in 2016, so missing means "no public record on file".

The list is saved to `models/audit_decisions.json` by the cell below.

In [10]:
# Record Decision B here so NB03 can read it (edit the list, then run)
# Decided 2026-10-31: the 2016-only block. mths_since_last_record is flagged by the >95% rule only because of
# 2012; from 2013 it is 81-88% missing every year (no public record = information), so it stays an attribute.
STRUCTURAL_GAP_FIELDS = ["all_util", "il_util", "inq_last_12m", "max_bal_bc", "mths_since_rcnt_il", "open_il_24m", "open_rv_24m"]
(MODELS / "audit_decisions.json").write_text(json.dumps({"structural_gap_fields": STRUCTURAL_GAP_FIELDS}, indent=1), encoding="utf-8")
print("saved", MODELS / "audit_decisions.json", "->", STRUCTURAL_GAP_FIELDS)

saved C:\Users\siava\OneDrive\Desktop\score-pilot-lab\models\audit_decisions.json -> ['all_util', 'il_util', 'inq_last_12m', 'max_bal_bc', 'mths_since_rcnt_il', 'open_il_24m', 'open_rv_24m']
